In [ ]:
from pathlib import Path
import pandas as pd

# Path to data folder (can change once if path ever moves)
DATA_DIR = Path.home() / '/Users/joshswan/Library/CloudStorage/redacted@example.com/My Drive/Retail Business Data'

# Function to find the latest files within the folders: returns latest file
def latest_file(folder, suffix):
    matches = sorted(folder.glob(f"*{suffix}"))
    if not matches:
        raise FileNotFoundError(f"No files matching *{suffix} in {folder}")
    return matches[-1]


# Each table lives in it's own folder:
folders={
    'customers': DATA_DIR / 'Dim_Customers',
    'orders': DATA_DIR / 'Fact_Orders',
    'products': DATA_DIR / 'Dim_Products',
    'line items': DATA_DIR / 'Dim_LineItems',
}

suffixes = {
    'customers': '_Customers.csv',
    'orders': '_Orders.csv',
    'products': '_Products.csv',
    'line items': '_Line Items.csv',
}

# Empty string to hold table names 
tables = {}

# Quick exploration of each table
for name, path in folders.items():
    df = pd.read_csv(latest_file(path, suffixes[name]))
    tables[name] = df
    print(f'=== {name.upper()} ===')
    print(df.shape)
    print(df.columns.tolist())
    df.info()

In [ ]:
# create dictionary for iteration of tables for grain verification
keys = {
    'customers': 'ID',
    'orders': 'ID',
    'products': 'ID', 
    'line items': 'ID'
}

# check the grain of the rows: what does 1 row represent? 
for name, col in keys.items():
    print(f"{name}:", tables[name][col].nunique(), "unique vs", len(tables[name]), "rows")


Hypotheses about the row counts:
    1. 

In [ ]:
# orders table exploration, cleaning of duplicates
tables['orders'].head(10)

orders = tables['orders'].drop_duplicates('Name')

orders.head(10)

total_revenue = orders['Shop total price'].sum()
total_orders = orders['Name'].nunique() # should equal total rows
total_customer_rows = len(orders['Name']) # should equal total orders

print(total_revenue)
print(f"{total_orders}", "unique orders vs", total_customer_rows, "rows")

- validated that orders table grain was showing one row per line item instead of one row per order
- removing duplicate 'Name' column items matched the grain to unique rows

In [ ]:
sum_of_nulls_orders = orders.isna().sum()
print("===== Columns with NULL values =====")
print(sum_of_nulls_orders)

missing_pct = orders.isna().mean().sort_values(ascending=False)
print("===== Missing Values Percentage per column =====")
print(missing_pct)

rows_fully_duplicated = orders.duplicated().sum()
print("===== Rows that are fully duplicated =====")
print(rows_fully_duplicated)

unique_values_per_column = orders.nunique()
print("===== Unique Values per Column =====")
print(unique_values_per_column)

In [ ]:
tables['customers'].isna().sum()